In [10]:
import os
import mne
import numpy as np
import glob


In [11]:
def load_hyp_annotations(hyp_path, sfreq=100):
    """Parses the binary .hyp file and returns an MNE Annotations object."""
    # Read binary data
    with open(hyp_path, 'rb') as f:
        raw_data = f.read()
    
    # Convert bytes to integers
    stages = np.frombuffer(raw_data, dtype=np.uint8)
    
    # Sleep-EDF .hyp files represent 30s epochs
    duration = 30.0
    onsets = np.arange(len(stages)) * duration
    
    # Map integers back to the string labels MNE expects or your LABEL_DICT
    # Note: We filter out 'Movement' (6) and 'Unscored' (7) if they exist
    inv_label_dict = {
        0: "Sleep stage W",
        1: "Sleep stage 1",
        2: "Sleep stage 2",
        3: "Sleep stage 3",
        4: "Sleep stage 4",
        5: "Sleep stage R"
    }
    
    description = [inv_label_dict.get(s, "Unknown") for s in stages]
    
    return mne.Annotations(onset=onsets, duration=[duration]*len(stages), 
                           description=description)

In [12]:

# Configuration
BASE_DATA_DIR = "/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/"
OUTPUT_DIR = "/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/processed/"
CHANNELS = ["Fpz-Cz"]
SAMPLING_RATE = 100

# Mapping labels to integers
LABEL_DICT = {
    "Sleep stage W": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,  # Merged N3 and N4
    "Sleep stage R": 4,
}

def get_file_pairs(data_dir):
    # Updated to look for .rec files instead of .edf for PSG
    psg_files = sorted(glob.glob(os.path.join(data_dir, "*.edf")))
    
    print(psg_files)
    # Hypnograms in older versions are still usually .edf or .lab
    hypno_files = sorted(glob.glob(os.path.join(data_dir, "*.hyp")))
    print("hypno: ", hypno_files)
    pairs = []
    for psg in psg_files:
        # Extract the ID (e.g., SC4001)
        print(os.path.basename(psg))
        file_id = os.path.basename(psg)[:6]
        matching_hypno = [h for h in hypno_files if file_id in h]
        
        if matching_hypno:
            pairs.append((psg, matching_hypno[0]))
            
    return pairs

def process_and_save(pairs):
    for psg_path, hypno_path in pairs:
        psg_filename = os.path.basename(psg_path)
        print(f"Processing: {psg_filename}")
            
        study_type = psg_filename[:2].upper() 
        subject_id = psg_filename[2:6]

        try:
            # MNE's read_raw_edf handles .rec files natively as they are EDF-compliant
            # infer_types=True helps with older EDF headers
            raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
            
            # Load Labels using the custom function
            if hypno_path.endswith('.hyp'):
                annot = load_hyp_annotations(hypno_path)
            else:
                annot = mne.read_annotations(hypno_path)
                
            raw.set_annotations(annot)

            # Now events_from_annotations will work as expected
            events, event_id = mne.events_from_annotations(
                raw, event_id=LABEL_DICT, chunk_duration=30, verbose=False
            )
            
            # Create epochs
            epochs = mne.Epochs(
                raw, events, event_id, tmin=0., tmax=30. - 1./SAMPLING_RATE, 
                baseline=None, preload=True, verbose=False
            )

            x = epochs.get_data() 
            y = epochs.events[:, 2]

            # Save logic
            save_dir = os.path.join(OUTPUT_DIR, study_type)
            os.makedirs(save_dir, exist_ok=True)
            
            # Replace .rec extension for the output filename
            output_name = psg_filename.replace(".rec", ".npz")
            output_file = os.path.join(save_dir, output_name)
            
            np.savez(output_file, x=x, y=y)
            
            print(f"Successfully processed: {study_type}/{subject_id} ({len(y)} epochs)")

        except Exception as e:
            print(f"Error processing {psg_filename}: {e}")

if __name__ == "__main__":
    pairs = get_file_pairs(BASE_DATA_DIR)
    print(pairs)
    if not pairs:
        print("No .rec/.edf pairs found. Check your directory path.")
    else:
        process_and_save(pairs)

['/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/sc4002e0.edf', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/sc4012e0.edf', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/sc4102e0.edf', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/sc4112e0.edf', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/st7022j0.edf', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/st7052j0.edf', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/st7121j0.edf', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF-20/physionet.org/files/sleep-edf/1.0.0/st7132j0.edf']
hypno:  ['/home/kasra/courses/2026-edge

/tmp/ipykernel_43150/3302248513.py:56: RuntimeWarning: Omitted 3343 annotation(s) that were outside data range.
  raw.set_annotations(annot)
/tmp/ipykernel_43150/3302248513.py:56: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annot)


Successfully processed: SC/4002 (2317 epochs)
Processing: sc4012e0.edf


/tmp/ipykernel_43150/3302248513.py:56: RuntimeWarning: Omitted 3363 annotation(s) that were outside data range.
  raw.set_annotations(annot)
/tmp/ipykernel_43150/3302248513.py:56: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annot)


Successfully processed: SC/4012 (2338 epochs)
Processing: sc4102e0.edf


...

Error processing st7052j0.edf: No matching events found for Sleep stage R (event id 4)
Processing: st7121j0.edf


/tmp/ipykernel_43150/3302248513.py:48: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
/tmp/ipykernel_43150/3302248513.py:56: RuntimeWarning: Omitted 1756 annotation(s) that were outside data range.
  raw.set_annotations(annot)
/tmp/ipykernel_43150/3302248513.py:56: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annot)


Successfully processed: ST/7121 (528 epochs)
Processing: st7132j0.edf


/tmp/ipykernel_43150/3302248513.py:48: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
/tmp/ipykernel_43150/3302248513.py:56: RuntimeWarning: Omitted 1670 annotation(s) that were outside data range.
  raw.set_annotations(annot)
/tmp/ipykernel_43150/3302248513.py:56: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annot)


Successfully processed: ST/7132 (386 epochs)
